# 1. Datenqualität prüfen

Zuerst schaue ich mir die Rohdaten an. Dabei prüfe ich den Aufbau, fehlende Werte, die Verteilung der Zielklassen und auffällige Angaben. Ich verändere hier noch nichts.

In [4]:
import pandas as pd

df = pd.read_csv("data/data.csv")

print("Zeilen und Spalten:", df.shape)
display(df.head())

# pd.read_csv lädt die CSV-Datei in die Tabelle df.
# df.shape zeigt die Anzahl der Zeilen und Spalten.
# df.head() zeigt die ersten fünf Personen.

Zeilen und Spalten: (19719, 23)


,N6,N8,N7,N1,N9,N10,N3,N5,E3,N2,...,E4,age,C4,E1,A4,E10,E9,gender,hand,target
0,1,1,1,1,1,1,2,1,5,5,...,2,53,1,4,5,1,5,Male,Right,Resilient
1,4,2,3,2,2,4,4,3,3,3,...,3,46,2,2,4,5,1,Female,Right,Moderate
2,5,5,5,5,5,5,5,5,1,1,...,4,14,1,5,5,1,5,Female,Right,Moderate
3,5,5,5,5,4,5,4,4,2,4,...,4,19,5,2,4,5,4,Female,Right,Overcontroller
4,3,3,3,3,3,4,3,3,3,3,...,3,25,3,3,5,5,3,Female,Right,Moderate


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19719 entries, 0 to 19718
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   N6      19719 non-null  int64
 1   N8      19719 non-null  int64
 2   N7      19719 non-null  int64
 3   N1      19719 non-null  int64
 4   N9      19719 non-null  int64
 5   N10     19719 non-null  int64
 6   N3      19719 non-null  int64
 7   N5      19719 non-null  int64
 8   E3      19719 non-null  int64
 9   N2      19719 non-null  int64
 10  E5      19719 non-null  int64
 11  N4      19719 non-null  int64
 12  E7      19719 non-null  int64
 13  E4      19719 non-null  int64
 14  age     19719 non-null  int64
 15  C4      19719 non-null  int64
 16  E1      19719 non-null  int64
 17  A4      19719 non-null  int64
 18  E10     19719 non-null  int64
 19  E9      19719 non-null  int64
 20  gender  19695 non-null  str  
 21  hand    19619 non-null  str  
 22  target  19719 non-null  str  
dtypes: int64(20), str(3)
m

In [7]:
# Fehlende Werte in jeder Spalte zählen
df.isna().sum()

N6          0
N8          0
N7          0
N1          0
N9          0
N10         0
N3          0
N5          0
E3          0
N2          0
E5          0
N4          0
E7          0
E4          0
age         0
C4          0
E1          0
A4          0
E10         0
E9          0
gender     24
hand      100
target      0
dtype: int64

### Erste Datenprüfung

Der Datensatz hat 19.719 Zeilen und 23 Spalten. In `gender` fehlen 24 Angaben, in `hand` sind es 100. Die übrigen Spalten sind vollständig. Bevor ich etwas bereinige, schaue ich mir die Werte genauer an.

In [8]:
# Zählen, wie oft jeder Persönlichkeitstyp vorkommt
df["target"].value_counts()

target
Moderate           8452
Resilient          6163
Overcontroller     2829
Undercontroller    2275
Name: count, dtype: int64

### Verteilung der Zielklassen

- **Moderate:** 8.452 Personen
- **Resilient:** 6.163 Personen
- **Overcontroller:** 2.829 Personen
- **Undercontroller:** 2.275 Personen

Die vier Persönlichkeitstypen kommen unterschiedlich häufig vor. Das berücksichtige ich dann später bei der Aufteilung in Trainings- und Testdaten.

In [9]:
# Überblick über das Alter: unter anderem kleinster und größter Wert
df["age"].describe()

count    1.971900e+04
mean     5.076703e+04
std      7.121272e+06
min      1.300000e+01
25%      1.800000e+01
50%      2.200000e+01
75%      3.100000e+01
max      1.000000e+09
Name: age, dtype: float64

### Prüfung der Altersspalte

Der Median liegt bei 22 Jahren. Der größte Wert ist 1.000.000.000 und kann kein echtes Alter sein. Der Durchschnitt fällt deshalb viel zu hoch aus. Als Nächstes schaue ich nach, wie viele solcher Werte vorkommen.

In [17]:
# Alterswerte vom größten zum kleinsten sortieren und die ersten zehn anzeigen
df["age"].sort_values(ascending=False).head(10)

4004     999999999
9987        412434
10715         2000
1649          1999
14628         1998
6614          1998
18661         1998
19614         1998
3016          1997
331           1997
Name: age, dtype: int64

### Größte Alterswerte

Unter den zehn größten Angaben stehen Werte wie 999.999.999 und 412.434. Auch 1997 bis 2000 können hier kein Alter in Jahren sein. Es könnten zwar Geburtsjahre sein. Sicher sagen kann ich das aber nicht. Deshalb rechne ich diese Werte nicht einfach um.

In [18]:
# Zählen, wie viele Altersangaben über 120 liegen
# 120 ist hier nur eine Grenze zur Untersuchung, noch keine Bereinigungsregel
(df["age"] > 120).sum()

np.int64(82)

### Anzahl auffälliger Alterswerte

Insgesamt liegen 82 von 19.719 Altersangaben über 120 Jahren. Mit dieser Grenze suche ich nur nach auffälligen Werten. Bereinigt habe ich an dieser Stelle noch nichts.

### Abgleich mit dem Codebook

Im Codebook steht, dass `age` das Alter in Jahren enthält. Die 82 Werte über 120 passen also nicht zur Beschreibung der Spalte. Da ich nicht weiß, was ursprünglich gemeint war, behandle ich sie später als fehlende Werte und rechne sie nicht um.

In [19]:
# N1: I get stressed out easily.
# Zählen, wie oft jeder Antwortwert bei dieser Frage vorkommt
df["N1"].value_counts()

N1
4    4949
3    4324
5    4308
2    3880
1    2257
0       1
Name: count, dtype: int64

### Antwortverteilung bei N1

Bei der Frage „I get stressed out easily“ reicht die Skala laut Codebook von 1 bis 5. Am häufigsten kommt der Wert 4 vor. Einmal steht dort aber auch eine 0, obwohl sie nicht zur Skala gehört. Deshalb schaue ich mir auch die anderen Fragen an.

In [20]:
# Alle Frage-Spalten prüfen; age, gender, hand und target sind keine Fragen
# Den kleinsten Antwortwert für jede Frage anzeigen
df.drop(columns=["age", "gender", "hand", "target"]).min()

N6     0
N8     0
N7     0
N1     0
N9     0
N10    0
N3     0
N5     0
E3     0
N2     0
E5     0
N4     0
E7     0
E4     0
C4     0
E1     0
A4     0
E10    0
E9     0
dtype: int64

### Kleinste Werte der Fragebewertungen

Bei allen 19 Fragen ist 0 der kleinste vorkommende Wert. Laut Codebook sind nur Werte von 1 bis 5 erlaubt. Als Nächstes prüfe ich, wie oft die 0 vorkommt und ob immer dieselbe Zeile betroffen ist.

In [21]:
# Für jede Frage zählen, wie oft der ungültige Antwortwert 0 vorkommt
# Die vier Nicht-Frage-Spalten werden nur für diese Prüfung ausgelassen
(df.drop(columns=["age", "gender", "hand", "target"]) == 0).sum()

N6     1
N8     1
N7     1
N1     1
N9     1
N10    1
N3     1
N5     1
E3     1
N2     1
E5     1
N4     1
E7     1
E4     1
C4     1
E1     1
A4     1
E10    1
E9     1
dtype: int64

### Häufigkeit ungültiger Fragewerte

Bei jeder der 19 Fragen kommt der Wert 0 genau einmal vor. Jetzt prüfe ich, ob diese Werte alle zu einer Person gehören oder auf mehrere Zeilen verteilt sind.

In [22]:
# Zählen, wie viele Zeilen bei mindestens einer Frage den Wert 0 haben
# any(axis=1) prüft dafür die Fragewerte innerhalb jeder Zeile
(df.drop(columns=["age", "gender", "hand", "target"]) == 0).any(axis=1).sum()

np.int64(1)

### Datensatz mit ungültigen Fragewerten

Alle 19 ungültigen Antworten gehören zur selben Zeile. Der komplette Fragebogen dieser Person ist damit nicht brauchbar. Deshalb entferne ich diese Zeile später aus meiner Arbeitskopie.

In [23]:
# Die größten Antwortwerte aller Frage-Spalten anzeigen, damit auch Werte über 5 ausschließen können.
# age, gender, hand und target gehören nicht zu den Fragen
df.drop(columns=["age", "gender", "hand", "target"]).max()

N6     5
N8     5
N7     5
N1     5
N9     5
N10    5
N3     5
N5     5
E3     5
N2     5
E5     5
N4     5
E7     5
E4     5
C4     5
E1     5
A4     5
E10    5
E9     5
dtype: int64

### Größte Werte der Fragebewertungen

Bei jeder Frage ist 5 der höchste Wert. Damit liegen die Werte am oberen Ende innerhalb der erlaubten Skala. Auffällig bleibt also nur die bereits gefundene Zeile mit den Nullwerten.

In [24]:
# Die Antwortverteilungen aller 19 Fragen zusammenfassen
# .T dreht die Tabelle: Jede Frage steht dadurch in einer eigenen Zeile
df.drop(columns=["age", "gender", "hand", "target"]).describe().T

,count,mean,std,min,25%,50%,75%,max
N6,19719.0,2.980374,1.320437,0.0,2.0,3.0,4.0,5.0
N8,19719.0,2.803235,1.350648,0.0,2.0,3.0,4.0,5.0
N7,19719.0,3.151935,1.299910,0.0,2.0,3.0,4.0,5.0
N1,19719.0,3.262082,1.308169,0.0,2.0,3.0,4.0,5.0
N9,19719.0,3.135250,1.298573,0.0,2.0,3.0,4.0,5.0
N10,19719.0,2.833764,1.313036,0.0,2.0,3.0,4.0,5.0
N3,19719.0,3.842690,1.138854,0.0,3.0,4.0,5.0,5.0
N5,19719.0,2.951722,1.272889,0.0,2.0,3.0,4.0,5.0
E3,19719.0,3.416755,1.236820,0.0,3.0,4.0,4.0,5.0
N2,19719.0,3.234596,1.177018,0.0,2.0,3.0,4.0,5.0


### Verteilung der Fragebewertungen

Für jede der 19 Fragen liegen 19.719 Antworten vor. Der Median liegt je nach Frage bei 3 oder 4. Die Durchschnittswerte reichen ungefähr von 2,63 bei `E1` bis 4,03 bei `A4`. Die Minimalwerte von 0 stammen aus der bereits gefundenen ungültigen Zeile.

In [25]:
# Häufigkeit aller gender-Werte anzeigen, auch fehlende Angaben
df["gender"].value_counts(dropna=False)

gender
Female    11985
Male       7608
Other       102
NaN          24
Name: count, dtype: int64

### Kategorien in `gender`

- **Female:** 11.985 Einträge
- **Male:** 7.608 Einträge
- **Other:** 102 Einträge
- **Fehlend:** 24 Einträge

Die vorhandenen Kategorien passen zum Codebook. Es fehlen nur 24 Angaben. Diese fülle ich erst beim Preprocessing auf.

In [26]:
# Häufigkeit aller hand-Werte anzeigen, auch fehlende Angaben
df["hand"].value_counts(dropna=False)

hand
Right    17424
Left      1724
Both       471
NaN        100
Name: count, dtype: int64

### Kategorien in `hand`

- **Right:** 17.424 Einträge
- **Left:** 1.724 Einträge
- **Both:** 471 Einträge
- **Fehlend:** 100 Einträge

Auch diese Kategorien passen zum Codebook. Die 100 fehlenden Angaben fülle ich erst beim Preprocessing auf.

## Punkt 1 abgeschlossen: Datenqualität geprüft

Ich habe zwei klare Probleme gefunden: einen komplett ungültigen Fragebogen und 82 unplausible Altersangaben. Außerdem fehlen einige Angaben in `gender` und `hand`. An den Rohdaten habe ich bisher nichts verändert.

# 2. Daten bereinigen

Für die Bereinigung arbeite ich mit der Kopie `df_clean`. Die ursprüngliche Tabelle `df` und die Datei `data/data.csv` bleiben unverändert.

In [27]:
# N1: I get stressed out easily.
# Die einzige Zeile mit N1 = 0 hat in allen 19 Fragen den ungültigen Wert 0
df_clean = df[df["N1"] != 0].copy()

# Kontrollieren, wie viele Zeilen vorher und nachher vorhanden sind
print("Vorher:", len(df), "Nachher:", len(df_clean))

Vorher: 19719 Nachher: 19718


### Bereinigung der Fragebewertungen

Ich habe die Zeile mit den 19 Nullwerten aus `df_clean` entfernt. Da der gesamte Fragebogen ungültig ist, kann ich die Antworten nicht sinnvoll ergänzen. Meine Arbeitskopie hat danach 19.718 statt 19.719 Zeilen.

In [28]:
# Auffällige Alterswerte in der bereinigten Arbeitskopie zählen
(df_clean["age"] > 120).sum()

np.int64(82)

### Auffällige Alterswerte nach der ersten Bereinigung

Auch nach dem Entfernen des ungültigen Fragebogens gibt es noch alle 82 Alterswerte über 120. Die entfernte Zeile gehörte also nicht zu diesen Fällen.

In [29]:
# Altersangaben über 120 Jahren in der Arbeitskopie als fehlend markieren
df_clean.loc[df_clean["age"] > 120, "age"] = pd.NA

# Kontrollieren: keine Werte über 120 mehr, dafür 82 fehlende Altersangaben
print("Über 120:", (df_clean["age"] > 120).sum())
print("Alter fehlt:", df_clean["age"].isna().sum())

Über 120: 0
Alter fehlt: 82


### Bereinigung auffälliger Alterswerte

Ich habe die 82 Altersangaben über 120 in `df_clean` als fehlend markiert. Die restlichen Angaben dieser Personen bleiben erhalten. Beim Preprocessing ersetze ich die fehlenden Alterswerte anhand der Trainingsdaten.

## Punkt 2 abgeschlossen: Daten bereinigt

Ich habe den ungültigen Fragebogen entfernt und die 82 unplausiblen Alterswerte als fehlend markiert. Die anderen fehlenden Angaben bleiben im Datensatz. Ich fülle sie erst beim Preprocessing auf.

# 3. Bereinigung kontrollieren und Daten speichern

Zum Schluss kontrolliere ich meine bereinigte Arbeitskopie und speichere sie als neue Datei.

In [30]:
# Fehlende Werte in den drei demografischen Spalten von df_clean zählen
df_clean[["age", "gender", "hand"]].isna().sum()

age        82
gender     24
hand      100
dtype: int64

### Verbleibende fehlende Werte

In `age` fehlen jetzt 82 Werte, in `gender` 24 und in `hand` 100. Die betroffenen Zeilen bleiben erhalten. Die Lücken fülle ich erst später beim Preprocessing auf.

In [31]:
# Bereinigte Arbeitskopie als neue CSV speichern; data.csv bleibt unverändert
df_clean.to_csv("data/data_processed.csv", index=False)

# Neue Datei testweise einlesen und Zeilen sowie Spalten kontrollieren
pd.read_csv("data/data_processed.csv").shape

(19718, 23)

### Bereinigten Datensatz gespeichert

Ich habe den bereinigten Datensatz unter `data/data_processed.csv` gespeichert. Er hat 19.718 Zeilen und 23 Spalten. Die Rohdatei `data/data.csv` bleibt unverändert.

Ich habe nur die Zeile mit den 19 ungültigen Antworten entfernt. Die 82 unplausiblen Alterswerte habe ich als fehlend markiert. Die fehlenden Angaben in `age`, `gender` und `hand` fülle ich später beim Preprocessing auf.

## Punkt 3 abgeschlossen: Bereinigung kontrolliert und Daten gespeichert

Die Datei `data/data_processed.csv` ist gespeichert. Damit kann ich jetzt mit der Modellierung weitermachen.